# v26 — расширяем смесь после победы v25

**Restart Kernel → Run All** в прежнем research `.venv`. 84 варианта, без обучения, GPU, интернета и ограничения времени. Доля R1: 40–100%. Контроль — конкретный v25 (50/50, k1=20, k2=3, λ=0.50).

Выбор только на calibration; на validation — контроль и один победитель. Кандидаты и порог фиксированы. Данные уже наблюдались; это development-поиск, не независимый тест. Автоматической смены MVP нет.


In [1]:
from pathlib import Path
import os, sys, subprocess
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/fusion_extension.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Markdown, display
from training import fusion_extension as experiment
SOURCE_RUN = 'map_search_v1'
RUN_NAME = 'fusion_extension_v1'
ALLOW_OUTER_EVALUATION = True
print('Kernel:', sys.executable, '\nRun:', RUN_NAME, '| experiments:', len(experiment.systems()))
print('CPU, cached features only; обучение: нет; кандидаты и порог R1: фиксированы')


Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Run: fusion_extension_v1 | experiments: 84
CPU, cached features only; обучение: нет; кандидаты и порог R1: фиксированы


## 1. Проверки компонентов


In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_fusion_extension.py', 'tests/test_map_search.py', 'tests/test_dual_role_inference.py'], check=True)


.................................                                        [100%]
33 passed in 4.92s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_fusion_extension.py', 'tests/test_map_search.py', 'tests/test_dual_role_inference.py'], returncode=0)

## 2. Проверка источников и фиксация сетки
Исторические runs не изменяются. При изменившихся исходниках/артефактах выполнение останавливается.


In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Output:', context['output'])
display(context['manifest']['fusion_extension_plan'])


Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_26_fusion_extension/runs/fusion_extension_v1


{'source_directory': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_25_map_search/runs/map_search_v1',
 'source_signature': '0d72bee94427ed4f3ad2cd718bd693d24123a048b94ac7075d029038c8733c71',
 'baseline': {'name': 'r1w50_k20_q3_l50',
  'k1': 20,
  'k2': 3,
  'lambda': 0.5,
  'r1_weight': 0.5},
 'reason': 'v25 winner reached the highest tested R1 weight; extend calibration search to 100%',
 'validation': 'current v25 baseline plus one calibration winner; no automatic promotion'}

## 3. Calibration → замороженный выбор → validation
Прогресс и время показываются для каждого варианта. Повторный запуск продолжает тот же RUN_NAME.


In [4]:
results = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)
print('Status:', results['status'], '| promoted:', results['promoted'])



CALIBRATION 1/84 | elapsed 10.1s
[START] calibration_r1w50_k20_q3_l50 | elapsed 0.00 h
calibration r1w50_k20_q3_l50: mAP=0.826410; R1 candidates unchanged
[DONE] calibration_r1w50_k20_q3_l50 | 0.0 min

CALIBRATION 2/84 | elapsed 10.5s
[START] calibration_r1w40_k10_q3_l40 | elapsed 0.00 h
calibration r1w40_k10_q3_l40: mAP=0.812436; R1 candidates unchanged
[DONE] calibration_r1w40_k10_q3_l40 | 0.0 min

CALIBRATION 3/84 | elapsed 10.9s
[START] calibration_r1w40_k10_q3_l50 | elapsed 0.00 h
calibration r1w40_k10_q3_l50: mAP=0.816213; R1 candidates unchanged
[DONE] calibration_r1w40_k10_q3_l50 | 0.0 min

CALIBRATION 4/84 | elapsed 11.4s
[START] calibration_r1w40_k10_q3_l65 | elapsed 0.00 h
calibration r1w40_k10_q3_l65: mAP=0.820675; R1 candidates unchanged
[DONE] calibration_r1w40_k10_q3_l65 | 0.0 min

CALIBRATION 5/84 | elapsed 11.8s
[START] calibration_r1w40_k10_q3_l75 | elapsed 0.00 h
calibration r1w40_k10_q3_l75: mAP=0.819868; R1 candidates unchanged
[DONE] calibration_r1w40_k10_q3_l75 

## 4. Результаты


In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
print('Reports and exports:', context['output'])


# v26 — расширение смеси после v25

Статус: complete
84 фиксированных варианта: доля R1 40/50/60/70/80/90/100%, k1 10/20/30, k2=3, λ 0.40/0.50/0.65/0.75.
Выбор только по calibration mAP; при равенстве сохраняется текущий v25 (50/50, 20/3/0.50).

| Validation: текущий v25 и выбор | mAP@10 | Rank-1 | F1 | TNR |
|---|---:|---:|---:|---:|
| r1w50_k20_q3_l50 | 0.828957 | 0.829960 | 0.795455 | 0.709677 |
| r1w80_k10_q3_l50 | 0.811639 | 0.813765 | 0.795455 | 0.709677 |

ΔmAP против v25: -0.017319. AP лучше/хуже/равно: 20/28/199.

## Calibration

| Вариант | mAP@10 |
|---|---:|
| r1w80_k10_q3_l50 | 0.841148 |
| r1w80_k10_q3_l65 | 0.840778 |
| r1w80_k10_q3_l40 | 0.839977 |
| r1w90_k10_q3_l40 | 0.839714 |
| r1w90_k10_q3_l50 | 0.839321 |
| r1w70_k20_q3_l50 | 0.838889 |
| r1w80_k10_q3_l75 | 0.838176 |
| r1w70_k10_q3_l65 | 0.837530 |
| r1w70_k20_q3_l40 | 0.837301 |
| r1w70_k20_q3_l65 | 0.837290 |
| r1w70_k10_q3_l50 | 0.837134 |
| r1w90_k10_q3_l65 | 0.836996 |
| r1w70_k10_q3_l40 | 0.835964 |
| r1w80_k20_q3_l65 | 0.835591 |
| r1w70_k10_q3_l75 | 0.834984 |
| r1w90_k10_q3_l75 | 0.834775 |
| r1w90_k20_q3_l50 | 0.834627 |
| r1w70_k20_q3_l75 | 0.833887 |
| r1w80_k20_q3_l40 | 0.833558 |
| r1w80_k20_q3_l50 | 0.833506 |
| r1w60_k10_q3_l50 | 0.833468 |
| r1w80_k30_q3_l65 | 0.832984 |
| r1w90_k20_q3_l40 | 0.832843 |
| r1w90_k20_q3_l65 | 0.832600 |
| r1w80_k30_q3_l50 | 0.832392 |
| r1w60_k10_q3_l40 | 0.832112 |
| r1w60_k10_q3_l65 | 0.831861 |
| r1w80_k20_q3_l75 | 0.831838 |
| r1w60_k10_q3_l75 | 0.830943 |
| r1w80_k30_q3_l75 | 0.830769 |
| r1w70_k30_q3_l50 | 0.830604 |
| r1w100_k30_q3_l50 | 0.830268 |
| r1w100_k30_q3_l40 | 0.829987 |
| r1w90_k20_q3_l75 | 0.829879 |
| r1w90_k30_q3_l65 | 0.829832 |
| r1w100_k10_q3_l75 | 0.829748 |
| r1w70_k30_q3_l65 | 0.829585 |
| r1w90_k30_q3_l50 | 0.829491 |
| r1w100_k30_q3_l65 | 0.829327 |
| r1w90_k30_q3_l75 | 0.829204 |
| r1w70_k30_q3_l75 | 0.828990 |
| r1w100_k10_q3_l65 | 0.828045 |
| r1w100_k30_q3_l75 | 0.827462 |
| r1w100_k20_q3_l40 | 0.827413 |
| r1w100_k20_q3_l50 | 0.827375 |
| r1w100_k10_q3_l50 | 0.826616 |
| r1w80_k30_q3_l40 | 0.826594 |
| r1w50_k20_q3_l50 | 0.826410 |
| r1w70_k30_q3_l40 | 0.826340 |
| r1w50_k20_q3_l65 | 0.825913 |
| r1w90_k30_q3_l40 | 0.825894 |
| r1w100_k20_q3_l65 | 0.825714 |
| r1w60_k30_q3_l65 | 0.825254 |
| r1w60_k30_q3_l50 | 0.824758 |
| r1w50_k30_q3_l65 | 0.824636 |
| r1w50_k10_q3_l65 | 0.824558 |
| r1w50_k30_q3_l50 | 0.824163 |
| r1w60_k20_q3_l65 | 0.824152 |
| r1w100_k20_q3_l75 | 0.823894 |
| r1w50_k10_q3_l50 | 0.823857 |
| r1w60_k30_q3_l40 | 0.823708 |
| r1w100_k10_q3_l40 | 0.823521 |
| r1w50_k30_q3_l40 | 0.823350 |
| r1w50_k10_q3_l75 | 0.823020 |
| r1w50_k10_q3_l40 | 0.822970 |
| r1w60_k20_q3_l50 | 0.821665 |
| r1w50_k20_q3_l40 | 0.821135 |
| r1w50_k20_q3_l75 | 0.821088 |
| r1w40_k10_q3_l65 | 0.820675 |
| r1w60_k30_q3_l75 | 0.820094 |
| r1w40_k10_q3_l75 | 0.819868 |
| r1w40_k20_q3_l65 | 0.819363 |
| r1w60_k20_q3_l40 | 0.819274 |
| r1w50_k30_q3_l75 | 0.818832 |
| r1w60_k20_q3_l75 | 0.818775 |
| r1w40_k30_q3_l65 | 0.816617 |
| r1w40_k10_q3_l50 | 0.816213 |
| r1w40_k30_q3_l40 | 0.816007 |
| r1w40_k30_q3_l50 | 0.815660 |
| r1w40_k20_q3_l75 | 0.815250 |
| r1w40_k20_q3_l50 | 0.815143 |
| r1w40_k30_q3_l75 | 0.814223 |
| r1w40_k10_q3_l40 | 0.812436 |
| r1w40_k20_q3_l40 | 0.809207 |

## Ограничения

Это следующий адаптивный development-поиск на уже наблюдавшихся данных, не независимый hidden test.
Сетка зафиксирована до нового запуска; validation только для v25 и одного calibration-победителя.
Никакого обучения, encoder inference, подбора порога, правок bbox или обмена между query.
Кандидаты/отказы/confidence неизменны; CSV кандидатов побайтно совпадает с v24/v25.
Реальные float32-эмбеддинги 2048, query затем gallery; отдельная конфигурация ranking, NPY replay проверен.
MVP не переключается автоматически. При проигрыше сохраняем конкретный v25, а не средний рецепт.
Источники неизменны: True. Время текущего вызова: 0.9 мин.


Reports and exports: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_26_fusion_extension/runs/fusion_extension_v1
